# Ancestor–Descendant CNV Comparison Analysis

This notebook compares copy number variation (CNV) profiles between ancestral and descendant samples to identify patterns of genomic gain and loss over evolutionary or experimental time.

## Overview

The workflow:
1. Loads CNV data (e.g., BED or processed coverage files)
2. Aligns genomic regions across samples
3. Compares ancestor vs descendant CNV states
4. Identifies gains, losses, and conserved regions
5. Aggregates results across multiple samples
6. Visualizes CNV differences and trends

## CNV Comparison Logic
Supports per-sample and aggregated comparisons. Classifies regions into categories:
  - **Gain**: increase in copy number relative to ancestor
  - **Loss**: decrease in copy number
  - **No Change**: stable copy number


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
from functools import reduce
from pathlib import Path

In [9]:
project_root = Path.cwd().parents[0]
input_files = str(project_root / "CNV" / "Coverage Files")
output_files = str(project_root / "CNV" / "Output")

with open('sd_samples.txt', 'r') as file:
    sd_ids = file.read().splitlines()

with open('nat_samples.txt', 'r') as file:
    nat_ids = file.read().splitlines()
sd_ids.remove("892B")
nat_ids.remove("ERR1308867")

In [2]:
# 1. Load chromosome lengths from .fai
chromosome_lengths = {}
fai_path = "newref.fasta.fai"
with open(fai_path, "r") as f:
    for line in f:
        fields = line.strip().split()
        chromosome_lengths[fields[0]] = int(fields[1])

In [3]:
# Create bins
def create_windows(cov, lengths, window_size):
    all_chrom_windows = []
    for chrom, length in lengths.items():
        current = cov.query("chrom == @chrom")

        bins = np.arange(0, length + window_size, window_size)
        current = current.assign(BIN=pd.cut(current["pos"], bins))
        windows = current.groupby(by="BIN", observed=True).agg({
            "chrom": "first",
            "pos": ["min", "max"],
            "coverage": ["min", "max", "mean", "median", "std"]
        })
        all_chrom_windows.append(windows)

    windowed_cov = pd.concat(all_chrom_windows)
    # Flatten multi-index columns
    windowed_cov.columns = ["_".join(c) for c in windowed_cov.columns]
    windowed_cov = windowed_cov.rename(columns={
        "chrom_first": "chrom",
        "pos_min": "start",
        "pos_max": "end",
        "coverage_min": "min",
        "coverage_max": "max",
        "coverage_mean": "mean",
        "coverage_median": "median",
        "coverage_std": "std"
    })
    windowed_cov["median"] = windowed_cov["median"].astype(int)
    windowed_cov["mean"] = windowed_cov["mean"].round(3)
    windowed_cov["std"] = windowed_cov["std"].round(3)
    windowed_cov = windowed_cov.reset_index(drop=True)
    return windowed_cov

In [ ]:
# Generate bins/windows for all coverage files
# FILES ALREADY EXIST - no need to run this again
coverage_files = glob.glob(f"{input_files}/*_coverage.txt")
for file in coverage_files:
    cov = pd.read_csv(file, sep="\t", header=None, names=["chrom", "pos", "cov"])
    cov["chrom"] = cov["chrom"].astype(str)
    cov = cov[cov["chrom"] != "chrM"]
    # Calculate median coverage (3rd column)
    median_cov = cov["cov"].median()
    cov["coverage_norm"] = cov["cov"] / median_cov
    windowed_cov = create_windows(
        cov.rename(columns={"coverage_norm": "coverage"}),
        chromosome_lengths,
        window_size=1000
    )
    base_name = Path(file).stem.replace("_coverage", "")
    output_csv = str(base_name) + "_coverage_values.txt"
    windowed_cov.to_csv(f"{input_files}/{output_csv}", index=False)
    print(f"Saved windowed coverage to: {output_csv}")

In [7]:
# Read sourdough and ancestor txt files
sd_ancestor_df = pd.read_csv(f"{input_files}/892B_coverage_values.txt")
nat_ancestor_df = pd.read_csv(f"{input_files}/ERR1308867_coverage_values.txt")

In [11]:
# Calculate Significant Gain and Loss of CNV between Ancestor and Descendant Samples
# Gain of CNV - CNV of Descendant is greater than Ancestor
# Loss of CNV - CNV of Ancestor is greater than Descendant
# Output is BED files in CNV_Gain_Loss folder

def calc_gains_and_losses(ids, ancestor, type):
    for samples in ids:
        cnv = pd.read_csv(f"{input_files}/{samples}" + "_coverage_values.txt")
        loss = f"{samples}_{type}_LOSS.bed"
        gains = f"{samples}_{type}_GAIN.bed"
        with open (f"{output_files}/CNV_Gain_Loss/{loss}", "w") as lose_CNV:
            with open(f"{output_files}/CNV_Gain_Loss/{gains}", "w") as gain_CNV:
                gain_CNV.write("Sample\tChromosome\tStart\tEnd\tAncestor\tSample\n")
                lose_CNV.write("Sample\tChromosome\tStart\tEnd\tAncestor\tSample\n")
                for i in range(0, len(ancestor)):
                    # If difference of ancestor and descendant CNV is greater than 0.5
                    # Loss of CNV from Ancestor to Descendant 
                    if(ancestor['mean'][i] - cnv['mean'][i] >= 0.5):
                        lose_CNV.write(f"{samples}\t{cnv['chrom'][i]}\t{cnv['start'][i]}\t{cnv['end'][i]}\t{ancestor['mean'][i]}\t{cnv['mean'][i]}\n")
                    # If difference of ancestor and descendant CNV is less than -0.5
                    # Gain of CNV from Ancestor to Descendant
                    elif(sd_ancestor_df['mean'][i] - cnv['mean'][i] <= -0.5):
                        gain_CNV.write(f"{samples}\t{cnv['chrom'][i]}\t{cnv['start'][i]}\t{cnv['end'][i]}\t{ancestor['mean'][i]}\t{cnv['mean'][i]}\n")
            gain_CNV.close()
        lose_CNV.close()     

calc_gains_and_losses(sd_ids, sd_ancestor_df, "SD")
calc_gains_and_losses(nat_ids, nat_ancestor_df, "NAT")

In [16]:
## Go through each BED file and note down start and end
# Read in each BED file as dataframe and merge on start, end, and ancestor CNV
def output_CNVs(pathway):
    df_list = []
    base_path = Path(project_root) / "CNV" / "Output" / "CNV_Gain_Loss"
    files = list(base_path.glob(f"*{pathway}.bed"))

    for file in files:
        sample_name = file.stem.replace(pathway, "").strip("_")
        df = pd.read_csv(
            file,
            sep="\t",
            names=["Sample", "Chromosome", "Start", "End", "Ancestor CNV", "Sample CNV"]
        )
        # Keep only essential columns, rename CNV column uniquely
        df = df[["Chromosome", "Start", "End", "Sample CNV"]].rename(
            columns={"Sample CNV": f"{sample_name}_CNV"}
        )
        df_list.append(df)

    # Merge all CNV columns side by side
    merged_df = df_list[0]
    for df in df_list[1:]:
        merged_df = pd.merge(
            merged_df,
            df,
            on=["Chromosome", "Start", "End"],
            how="outer"
        )

    return merged_df

In [ ]:
# Merged CNV datasets across samples
# Annotated regions with gain/loss classifications
nat_gain = output_CNVs('_NAT_GAIN')
nat_loss = output_CNVs('_NAT_LOSS')
sd_gain = output_CNVs('_SD_GAIN')
sd_loss = output_CNVs('_SD_LOSS')

nat_gain.to_excel(f'{output_files}/combined_nat_gain.xlsx', header=True, index=False)
sd_gain.to_excel(f'{output_files}/combined_sd_gain.xlsx', header=True, index=False)
nat_loss.to_excel(f'{output_files}/combined_nat_loss.xlsx', header=True, index=False)
sd_loss.to_excel(f'{output_files}/combined_sd_loss.xlsx', header=True, index=False)